# Read Job Description Data From MySQL

This notebook reads Job Description (JD) data directly from the database.
Connection settings are loaded from `back-end/.env` through `app.db`.

In [6]:
from pathlib import Path
import sys

import pandas as pd
from sqlalchemy import text

import json

# Ensure ai-service root is importable from the notebook folder.
cwd = Path.cwd().resolve()
if (cwd / "app").exists():
    ai_service_root = cwd
elif (cwd.parent / "app").exists():
    ai_service_root = cwd.parent
else:
    raise FileNotFoundError("Cannot locate ai-service root directory containing app/.")

if str(ai_service_root) not in sys.path:
    sys.path.insert(0, str(ai_service_root))

from app.db import engine, test_connection  # noqa: E402

assert test_connection(), "DB connection failed. Check back-end/.env and MySQL service."

✅ Kết nối DB thành công!


In [7]:
limit = 100

sql = text("""
SELECT
    title,
    description,
    requirements,
    responsibilities,
    qualifications,
    experience_level
FROM Job_Descriptions
WHERE job_id = 5
ORDER BY job_id DESC
LIMIT :limit
""")

with engine.connect() as connection:
    jd_df = pd.read_sql(sql, connection, params={"limit": limit})

print(f"Loaded {len(jd_df)} rows.")

jd_json = jd_df.to_dict(orient="records")
multiline_fields = ["requirements", "responsibilities", "qualifications"]

for item in jd_json:
    for field in multiline_fields:
        value = item.get(field)
        if isinstance(value, str):
            item[field] = [line.strip() for line in value.splitlines() if line.strip()]

jd_output_path = ai_service_root / "notebooks" / "jd.json"
jd_output_path.write_text(
    json.dumps(jd_json, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print(f"Saved JSON to: {jd_output_path}")
print(json.dumps(jd_json, ensure_ascii=False, indent=2, default=str))

Loaded 1 rows.
Saved JSON to: D:\datn\source_code\ai-service\notebooks\jd.json
[
  {
    "title": "IT Comtor tiếng Hàn",
    "description": "Thực hiện phiên dịch (Hàn - Việt, Việt - Hàn) trong các cuộc họp trực tiếp hoặc trực tuyến giữa đội ngũ phát triển phần mềm (Dev team) và khách hàng Hàn Quốc.\nBiên dịch các tài liệu kỹ thuật, tài liệu dự án và báo cáo chuyên ngành từ tiếng Hàn sang tiếng Việt và ngược lại.\nHỗ trợ phiên dịch cho khách hàng Hàn Quốc trong các chuyến công tác tại Việt Nam.\nPhối hợp và hỗ trợ quản lý dự án trong việc theo dõi tiến độ và giao tiếp với các bên liên quan.\nThực hiện các công việc hành chính và báo cáo khác theo yêu cầu của cấp trên để phục vụ hoạt động của bộ phận.",
    "requirements": [
      "Tốt nghiệp Đại học chuyên ngành Ngôn ngữ Hàn Quốc hoặc các chuyên ngành có liên quan.",
      "Có kinh nghiệm làm việc thực tế tại các công ty Hàn Quốc hoặc trong lĩnh vực IT Comtor (ưu tiên ứng viên có từ 3 năm kinh nghiệm trở lên).",
      "Thành thạo tin họ